In [9]:
!pip install pytorch-forecasting --extra-index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu


In [10]:
import numpy as np
import pandas as pd
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.models import BaseModel, BaseModelWithCovariates, Baseline

In [11]:
class ZillowHousingDataService:
    
    def __init__(self, new_construction_price_per_square_foot_data_set_empty={}):
        self.new_construction_price_per_square_foot_data_set_formatted = new_construction_price_per_square_foot_data_set_empty
        
    def get_new_construction_price_per_square_foot_data_set_formatted(self):
        return self.new_construction_price_per_square_foot_data_set_formatted
        
    def get_new_construction_price_per_square_foot_data_set_training(self, region_id):#, indices):
        return self.new_construction_price_per_square_foot_data_set_formatted[region_id]#.take(indices)
        
    def format_new_construction_price_per_square_foot_data_set(self, create_repeating_array_function, data_frames, region_ids=None):
        
        if region_ids == None:
            index_values = data_frames.index
            
        else:
            index_values = region_ids
            
        for index_value in index_values:
            
            new_data_frame = data_frames.loc[[index_value]]
            
            time_series_data = new_data_frame.get(data_frames.columns[4:len(data_frames.columns)])
            
            new_data_frame.drop(columns=new_data_frame.columns, inplace=True)
            
            new_data_frame = new_data_frame.assign(RegionID=[create_repeating_array_function(index_value, len(time_series_data.columns))])
            
            new_data_frame = new_data_frame.explode("RegionID")
            
            new_data_frame.reset_index(drop=True, inplace=True)
            
            new_data_frame.insert(loc=0, column="Index", value=new_data_frame.index, allow_duplicates=False)
            
            new_data_frame = new_data_frame.assign(Date=time_series_data.columns.T)
            
            new_data_frame = new_data_frame.assign(Price=time_series_data.T.to_numpy(copy=True))
            
            self.new_construction_price_per_square_foot_data_set_formatted[index_value] = new_data_frame

In [12]:
data_file_path = "~/ece492-final-project/"
data_file_name = "Metro_new_con_median_sale_price_per_sqft_uc_sfrcondo_month.csv"

data_frames = pd.read_csv(data_file_path + data_file_name)
data_frames.set_index("RegionID", inplace=True)
data_frames.sort_index(inplace=True)
#print(data_frames)

In [13]:
data_service = ZillowHousingDataService()
data_service.format_new_construction_price_per_square_foot_data_set(np.repeat, data_frames) #data_frames.index[0:1:1]
formatted_data_sets = data_service.get_new_construction_price_per_square_foot_data_set_formatted()

#print(formatted_data_sets)
#print(formatted_data_sets[102001])

In [14]:
# get data sets with take first half of indices
#data = data_service.get_new_construction_price_per_square_foot_data_set_training(102001, np.arange(0, 50))
data = formatted_data_sets[102001]
#print(data)

In [15]:
training = TimeSeriesDataSet(
    data=data,
    time_idx="Index",
    target="Price",
    group_ids=["RegionID"]
)

In [8]:
batch_size = 128
validation = TimeSeriesDataSet.from_dataset(training, data, min_prediction_idx=training.index.time.max() + 1, stop_randomization=True)
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=2)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size, num_workers=2)
